# The role-tagged `Panel`

A `Panel` is a units × time table plus a `RoleMap` that says which column is the unit index,
which is time, and which entity each measured column is. **A column adopts the dimension of
its role** (review D3): units are declared once, here, and nothing else in an analysis
mentions them.

The panel validates and *reports*; it never imputes.

In [ ]:
import numpy as np
import pandas as pd

from axiom.core import Covariate, D, Outcome, Treatment, dimensionless
from axiom.data import Completeness, Panel, PanelError, RoleKind, RoleMap

In [ ]:
rng = np.random.default_rng(7)
units, periods = ["p01", "p02", "p03", "p04"], range(12)
df = pd.DataFrame(
    {
        "plot": np.repeat(units, len(periods)),
        "week": np.tile(list(periods), len(units)),
        "yield_kg": rng.gamma(5.0, 40.0, 48),
        "fert_usd": rng.uniform(0, 200, 48),
        "irrig_usd": rng.uniform(0, 80, 48),
        "rain_z": rng.normal(0, 1, 48),
    }
).sample(frac=1, random_state=1)  # shuffled on purpose; Panel sorts
df.head()

In [ ]:
roles = RoleMap(
    unit="plot",
    time="week",
    outcome=("yield_kg", Outcome(name="yield_total", dimension=D.outcome, unit="kg")),
    treatments={
        "fert_usd": Treatment(name="fertilizer", dimension=D.currency, unit="USD"),
        "irrig_usd": Treatment(name="irrigation", dimension=D.currency, unit="USD"),
    },
    covariates={"rain_z": Covariate(name="rainfall", dimension=dimensionless())},
)
print(roles.columns)
print(roles.measured)
kind: RoleKind = roles.kind_of("fert_usd")
print(kind, roles.dimension_of("fert_usd"), roles.unit_of("yield_kg"))

In [ ]:
panel = Panel(df, roles)
panel

In [ ]:
print(panel.units)
print(panel.periods[:5], "...")
print(panel.frame.head(3))

## Shapes for estimators

`column` gives the long vector; `array` gives `(n_units, n_periods)` with NaN where a cell
is absent; `wide` is the labelled version.

In [ ]:
print(panel.column("yield_kg").shape, panel.array("fert_usd").shape)
panel.wide("irrig_usd").iloc[:, :4]

## Completeness is reported, not fixed (review D5)

In [ ]:
c: Completeness = panel.completeness()
print(c)

In [ ]:
holey = Panel(df.iloc[3:], roles)            # drop three rows somewhere
c = holey.completeness()
print(c.balanced, c.missing_cells, c.gaps)
print("NaNs in the wide array:", int(np.isnan(holey.array("yield_kg")).sum()))

try:
    holey.require_balanced(context="a synthetic-control estimator")
except PanelError as e:
    print("refused:", e)

## Validation failures are loud

In [ ]:
for bad in (
    lambda: Panel(df.drop(columns=["rain_z"]), roles),
    lambda: Panel(df.assign(extra=1.0), roles),
    lambda: Panel(df.assign(yield_kg="n/a"), roles),
):
    try:
        bad()
    except PanelError as e:
        print("PanelError:", e)

## Identity

`content_hash()` covers the roles and the data, is row-order invariant, and is what `io`
records in provenance.

In [ ]:
print(panel.content_hash()[:16] == Panel(df.sample(frac=1, random_state=3), roles).content_hash()[:16])
print(panel.select_units(["p02", "p04"]).units)